## Stage 4, piece 1: independent-draw simulation

**What this does:** for each of the six portfolios, we roll the dice 10,000 times.
Each roll is one simulated year. In a year, every funded loan either defaults or
survives, decided by its own default risk, independently of the others. We collect
interest on survivors, subtract losses on defaulters, and get that year's return
per dollar. Ten thousand years gives a spread of outcomes per portfolio.

**Independent means each loan's coin flip is on its own.** No shared economy yet.
This is the simple baseline. Its known weakness: with thousands of loans flipping
independently, the good and bad average out, so almost every year lands near the
expected return and the tail is thin. That is expected. Piece 2 adds the shared
economy that makes bad years cluster.

**The three metrics:**
- average return: the typical year. Should land near the LP's expected return.
- spread: how much returns bounce year to year. The simplest risk measure.
- bad year (5th percentile): the return in a year worse than 95% of years. First
  look at the downside.

**Fractional loans handled proportionally:** a loan funded at 40% contributes 40%
of its interest and 40% of its loss.

**LGD is 30%,** the Sirignano base case, already in the `loss_if_default` column.

In [1]:
import polars as pl
import numpy as np
from pathlib import Path

PROC = Path("../data/processed")
port = pl.read_parquet(PROC / "scaffold_portfolios.parquet")

pd_arr = port["pd_catboost"].to_numpy()
interest = port["interest_income_7yr"].to_numpy()
loss = port["loss_if_default"].to_numpy()
upb = port["ORIG_UPB"].to_numpy()

port_cols = [c for c in port.columns if c.startswith("x__")]

N_SIMS = 10_000
rng = np.random.default_rng(591)

# one big matrix of coin flips, reused across portfolios so they see the same years
# defaults[s, i] = True if loan i defaults in simulated year s
defaults = rng.random((N_SIMS, len(pd_arr))) < pd_arr

print(f"{'score':9} {'rule':13} {'avg %':>8} {'spread':>8} {'bad yr %':>9}")
results = {}
for col in port_cols:
    x = port[col].to_numpy()
    funded_upb = (x * upb).sum()

    # per year: interest on all funded loans, minus loss on the ones that defaulted
    interest_total = (x * interest).sum()
    loss_per_year = defaults @ (x * loss)          # length N_SIMS
    ret_per_year = (interest_total - loss_per_year) / funded_upb

    avg = ret_per_year.mean()
    spread = ret_per_year.std()
    bad = np.percentile(ret_per_year, 5)
    results[col] = ret_per_year

    sc, rule = col.replace("x__", "").split("__")
    print(f"{sc:9} {rule:13} {avg*100:>7.2f} {spread*100:>7.3f} {bad*100:>8.2f}")

score     rule             avg %   spread  bad yr %
FICOxLTV  risk-sort       25.30   0.016    25.27
FICOxLTV  greedy-return   30.72   0.035    30.66
FICOxLTV  LP              30.51   0.033    30.45
CatBoost  risk-sort       24.73   0.009    24.72
CatBoost  greedy-return   30.89   0.025    30.85
CatBoost  LP              30.64   0.023    30.61


## Finding: independent draws are unrealistically calm

Baseline simulation, 10,000 years, each loan defaulting on its own.

| Score | Rule | avg % | spread | bad year % |
|---|---|---|---|---|
| CatBoost | greedy-return | 30.89 | 0.025 | 30.85 |
| CatBoost | LP | 30.64 | 0.023 | 30.61 |
| FICO×LTV | greedy-return | 30.72 | 0.035 | 30.66 |
| FICO×LTV | LP | 30.51 | 0.033 | 30.45 |
| FICO×LTV | risk-sort | 25.30 | 0.016 | 25.27 |
| CatBoost | risk-sort | 24.73 | 0.009 | 24.72 |

**The spread is nearly zero.** Every simulated year lands within a few hundredths
of a percent of the average. The bad year sits right on top of the average
everywhere. Nothing bad ever happens.

**Why:** when thousands of loans each flip their own coin, the good and bad cancel
out. The law of averages crushes the year-to-year variation to almost nothing.

**Two checks pass:**
- Average returns match the LP's expected returns, so the simulation is wired
  right.
- The ordering is sensible: greedy-return and LP earn most, risk-sort least.

**Why this matters:** independent draws hide the entire point of the project. The
LP's spread across states and greedy-return's concentration look identical here,
because independence cannot see that loans default together in bad years. Piece 2
adds a shared economy so bad years cluster, which is where the portfolios finally
separate.